# AURA Motor Model — Research Notebooks

These notebooks support the motor-skills modeling pipeline:

- **Dataset**: one row per session (your final `motor_sessions.csv`)
- **Goal**: train a *hostable* model that outputs an `impairment_score` in **[0,1]**
  - **0** = better motor performance in this task
  - **1** = more assistance needed in this task
- **Leakage control**: splits are grouped by **userId** (no user appears in train and test)

> ⚠️ This is a functional interaction score for this game task, **not a medical diagnosis**.


## 04 — Scoring demo & JSON output

Loads artifacts and scores sessions by row and by userId.

In [1]:
import os, subprocess, sys

DATASET = os.environ.get("AURA_DATASET", "../datasets/final/motor_sessions.csv")
OUTDIR  = os.environ.get("AURA_OUTDIR", "../model_registry/motor/1.0.0")

print("DATASET:", DATASET)
print("OUTDIR :", OUTDIR)


DATASET: ../datasets/final/motor_sessions.csv
OUTDIR : ../model_registry/motor/1.0.0


In [2]:
cmd = [sys.executable, "../training/score_one_session.py",
       "--csv", DATASET,
       "--outdir", OUTDIR,
       "--row", "0"]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Scoring failed")


{
  "sessionId": "session_1767637948400_lwki0nm0z2",
  "userId": "BVK",
  "motor_profile": {
    "impairment_score": 0.343,
    "confidence": 0.7972,
    "latent_score": -11.6459,
    "data_quality": {
      "imputed_numeric_fields": 53
    },
    "debug": {
      "impairment_score_A": 0.4444,
      "impairment_score_B1": 0.3227,
      "impairment_score_B2": 0.343,
      "used": "B2",
      "agreement_abs_diff_B2_vs_A": 0.1014
    }
  },
  "notes": [
    "Not a medical diagnosis.",
    "impairment_score is a functional interaction score in this task (0=better, 1=more assistance needed)."
  ]
}



In [3]:
import pandas as pd
df = pd.read_csv(DATASET)
uid = str(df["userId"].astype(str).iloc[0])
print("Using userId:", uid)

cmd = [sys.executable, "../training/score_one_session.py",
       "--csv", DATASET,
       "--outdir", OUTDIR,
       "--userId", uid,
       "--userSession", "latest"]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Scoring failed")


Using userId: BVK
{
  "sessionId": "session_1767637948400_lwki0nm0z2",
  "userId": "BVK",
  "motor_profile": {
    "impairment_score": 0.343,
    "confidence": 0.7972,
    "latent_score": -11.6459,
    "data_quality": {
      "imputed_numeric_fields": 53
    },
    "debug": {
      "impairment_score_A": 0.4444,
      "impairment_score_B1": 0.3227,
      "impairment_score_B2": 0.343,
      "used": "B2",
      "agreement_abs_diff_B2_vs_A": 0.1014
    }
  },
  "notes": [
    "Not a medical diagnosis.",
    "impairment_score is a functional interaction score in this task (0=better, 1=more assistance needed)."
  ]
}



## Extension-side JSON (recommended)

```json
{
  "sessionId": "...",
  "userId": "...",
  "motor_profile": {
    "impairment_score": 0.63,
    "confidence": 0.78,
    "latent_score": -1.12
  },
  "notes": [
    "Not a medical diagnosis.",
    "Functional interaction score in this task (0=better, 1=more assistance needed)."
  ]
}
```
